In [1]:
import re

In [11]:
property_pattern = re.compile(
        r'(?P<INDENT>\s*)(?P<NAME>self.\w+) = (?P<VALUE>None|list\(\)) # (of )?(?P<TYPE>[\w\.]+)\n'
    )

In [13]:
from IdlParser import IdlParser

In [16]:
# listing the classes defined under IdlParser class
parser_subclasses = [cls for cls in dir(IdlParser) if isinstance(getattr(IdlParser, cls), type)]

In [18]:
new_content = []
with open('IdlParser.py', 'r') as f:
    for line in f.readlines():
        # add optional to typing imports
        if line == "\tfrom typing import TextIO\n":
            new_content.append("\tfrom typing import TextIO, Optional, List\n")
            continue
        if line == "\tfrom typing.io import TextIO\n":
            new_content.append(line)
            new_content.append("\tfrom typing import Optional, List\n")
            continue
        if match := property_pattern.match(line):
            indent = match.group('INDENT')
            name = match.group('NAME')
            value = match.group('VALUE')
            type_ = match.group('TYPE')
            if value == 'None':
                type_ = type_ if type_ not in parser_subclasses else f'IdlParser.{type_}'
                new_line = f"{indent}{name}: Optional[{type_}] = None\n"
            else:
                stripped_type = type_[:-1]
                type_ = f'IdlParser.{stripped_type}' if stripped_type in parser_subclasses else type_
                new_line = f"{indent}{name}: List[{type_}] = list()\n"
            new_content.append(new_line)
            continue
        new_content.append(line)
with open('IdlParser2.py', 'w') as f:
    f.writelines(new_content)